# Get to Know a Dataset: FoMo — Forêt Montmorency Multi-Season Robot Navigation Dataset

This notebook is a guided tour of the **FoMo (Forêt Montmorency) dataset**, a comprehensive multi-season robotic data collection recorded over one year in a boreal forest. More usage examples, tutorials, and documentation for this dataset can be found on the [dataset website](https://fomo.norlab.ulaval.ca).

The FoMo dataset was recorded between November 2024 and November 2025 at Forêt Montmorency, a boreal forest located 80 km north of Quebec City, Canada. It features **over 64 km of diverse trajectories** repeated across **12 deployments** throughout the year, covering dramatic seasonal changes including snow accumulation exceeding 1 m.

**Sensors included:** two lidars (rotating and hybrid solid-state), one FMCW radar, a stereo camera, a wide-lens monocular camera, two IMUs, two microphones, and on-site meteorological instrumentation.

**Primary use cases:** odometry evaluation, SLAM benchmarking, place recognition, terrain classification, traversability estimation, and semantic segmentation under seasonal change.

The **FoMo SDK** provides dedicated tutorials for each sensor modality:
- 🐍 **Python tutorials:** [Audio](https://github.com/norlab-ulaval/fomo-sdk/blob/main/examples/audio_example.ipynb), [Image](https://github.com/norlab-ulaval/fomo-sdk/blob/main/examples/image_example.ipynb), [Lidar](https://github.com/norlab-ulaval/fomo-sdk/blob/main/examples/lidar_example.ipynb), [Radar](https://github.com/norlab-ulaval/fomo-sdk/blob/main/examples/radar_example.ipynb)
- 🦀 **Rust tutorials:** available at [https://github.com/norlab-ulaval/fomo-sdk](https://github.com/norlab-ulaval/fomo-sdk)

> **Citation:** Boxan et al. (2026). *FoMo: A Multi-Season Dataset for Robot Navigation in Forêt Montmorency.* arXiv:2603.08433 [cs.RO]. https://arxiv.org/abs/2603.08433

---
## Q1: How is the dataset organized?

The FoMo dataset is hosted in the public S3 bucket **`fomo-dataset`**. All data lives under the top-level `data/` prefix, then organized in a two-level hierarchy:

1. **Deployment folder** — identified by the recording date: `data/YYYY-MM-DD/`
2. **Session (trajectory) folder** — identified by the trajectory color and start time: `data/<color>-YYYY-MM-DD-HH-mm/`

The six trajectory colors are: **Red**, **Blue**, **Green**, **Magenta**, **Orange**, and **Yellow**, each representing a distinct route within the forest.

Within each session folder, data is organized as follows:

```
data/
└── YYYY-MM-DD/
    └── <color>-YYYY-MM-DD-HH-mm/
    ├── audio_left/             → 1-second .wav snippets (16 kHz)
    ├── audio_right/            → 1-second .wav snippets (16 kHz)
    ├── basler/                 → Monocular wide-lens images, 1920×1200 px (.png)
    ├── calib/                  → Sensor intrinsics & extrinsics (.json)
    │   ├── basler.json
    │   ├── imu.json
    │   ├── transforms.json
    │   ├── zedx_left.json
    │   └── zedx_right.json
    ├── leishen/                → Leishen LS128S1 lidar point clouds (.bin)
    ├── navtech/                → Navtech CIR-304H radar polar scans (.png)
    ├── robosense/              → RoboSense Ruby Plus lidar point clouds (.bin)
    ├── zedx_left/              → Left stereo camera images, 1920×1200 px (.png)
    ├── zedx_right/             → Right stereo camera images, 1920×1200 px (.png)
    ├── metadata/               → 16 .csv files (power, weather, motors, etc.)
    │   ├── basler_metadata.csv
    │   ├── battery_logs.csv
    │   ├── cmd_velocity_left.csv
    │   ├── cmd_velocity_right.csv
    │   ├── cmd_velocity.csv
    │   ├── current_left.csv
    │   ├── current_right.csv
    │   ├── dps310.csv
    │   ├── meteo_data.csv
    │   ├── snow_data.csv
    │   ├── velocity_left.csv
    │   ├── velocity_right.csv
    │   ├── vn100_pressure.csv
    │   ├── vn100_temperature.csv
    │   ├── voltage_left.csv
    │   └── voltage_right.csv
    ├── gt.txt                  → Ground truth trajectory (TUM format)
    ├── gt_covariance.csv       → Per-point GT covariances from PPK-GNSS
    ├── odom.csv                → Wheel odometry
    ├── vectornav.csv           → VectorNav VN-100 IMU data
    └── xsens.csv               → Xsens MTi-30 IMU data
```

---
## 1. Requirements & Setup

This notebook requires the following Python libraries. Install them using pip if needed:

```
# requirements.txt
# boto3 >= 1.38.0
# pandas >= 2.0.0
# numpy >= 1.24.0
# matplotlib >= 3.7.0
```

Because FoMo is a **public** dataset, no AWS credentials are required. We use `boto3` with the `UNSIGNED` signature configuration.

In [ ]:
# Built-in libraries
import io
import json
from pprint import pprint

# Third-party libraries
import boto3
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm
from botocore import UNSIGNED
from botocore.config import Config

# ── S3 client (unsigned = public access, no AWS account required) ──────────
BUCKET = "fomo-dataset"
DATA_PREFIX = "data/"  # All FoMo data lives under this top-level prefix
s3 = boto3.client("s3", config=Config(signature_version=UNSIGNED))

print("Setup complete. Connected to S3 bucket:", BUCKET)
print("Data prefix:", DATA_PREFIX)

### Listing deployments and sessions

Let's first explore the top-level structure of the bucket to see all available deployments.

In [ ]:
def list_prefixes(prefix="", delimiter="/"):
    """List all common prefixes (i.e., 'folders') under a given prefix."""
    resp = s3.list_objects_v2(Bucket=BUCKET, Prefix=prefix, Delimiter=delimiter)
    return [p["Prefix"] for p in resp.get("CommonPrefixes", [])]

# Top-level: deployment dates (under data/ prefix)
deployments = list_prefixes(prefix=DATA_PREFIX)
print(f"Found {len(deployments)} deployments:\n")
for d in deployments:
    print(" ", d)

In [ ]:
# For a given deployment, list all trajectory sessions
example_deployment = "data/2025-01-29/"
sessions = list_prefixes(prefix=example_deployment)

print(f"Sessions in deployment '{example_deployment}':\n")
for s in sessions:
    print(" ", s)

---
## Q2: What data formats are present?

The FoMo dataset uses several standard formats, each chosen to best represent the sensor modality:

| Format | Sensor / Data | Notes |
|--------|--------------|-------|
| `.bin` | Lidar point clouds (Leishen & RoboSense) | Binary float32 arrays: `[x, y, z, intensity, ring_id, timestamp]` per point |
| `.png` | Camera images (Basler, ZED X left/right) | Rectified/undistorted, 1920×1200 px |
| `.png` | Radar scans (Navtech) | Encoded polar images: M azimuths × (11 + R) columns, following Oxford conventions |
| `.wav` | Audio (left/right microphones) | 1-second mono snippets at 16 kHz |
| `.csv` | IMU, odometry, metadata, ground truth covariances | Standard comma-separated, timestamped in UNIX microseconds |
| `.txt` | Ground truth trajectory (`gt.txt`) | TUM format: `t x y z qx qy qz qw` (timestamps in seconds) |
| `.json` | Sensor calibration | Intrinsics and extrinsics for all sensors |

### Why these formats?

**Binary `.bin` for lidar:** Raw float32 binary is compact, fast to read with `numpy`, and avoids the overhead of text parsing for large point clouds (tens of thousands of points per scan at 10 Hz).

**PNG for radar:** The Navtech radar's polar scan is encoded as a 2D image following the Oxford RobotCar convention, making it directly usable with existing radar processing libraries. The SDK provides polar-to-Cartesian conversion utilities.

**TUM format for ground truth:** The TUM format is a widely adopted community standard for trajectory evaluation in SLAM benchmarks, ensuring compatibility with common evaluation tools such as `evo`.

**CSV for time-series metadata:** Lightweight, human-readable, and directly importable with `pandas` for quick exploration.

The SDK available at [fomo.norlab.ulaval.ca](https://fomo.norlab.ulaval.ca) provides Python and Rust scripts to convert between human-readable formats and ROS2 `mcap` rosbags for users preferring a ROS2 workflow.

> 💡 **SDK Tutorials:** Step-by-step tutorials for loading and manipulating each data modality are available in the SDK:
> - **Python:** Audio, Image, Lidar, and Radar data examples
> - **Rust:** Conversion scripts (`.bin` ↔ `.csv` for lidar, polar ↔ Cartesian for radar) and ROS2 mcap generation
>
> See [https://github.com/norlab-ulaval/fomo-sdk](https://github.com/norlab-ulaval/fomo-sdk)

---
## Q3: How do I download and load data?

### 3a. Loading CSV metadata (weather, power, velocities)

The `metadata/` directory in each session contains 16 CSV files. Let's load and explore the meteorological and snow data from the cold January 2025 deployment.

In [ ]:
def get_csv(key):
    """Download and parse a CSV file from S3 into a pandas DataFrame."""
    obj = s3.get_object(Bucket=BUCKET, Key=key)
    return pd.read_csv(io.BytesIO(obj["Body"].read()))

# ── Example: January 29, 2025 – Green trajectory ──────────────────────────
# Dynamically find the green session to avoid hardcoding the exact folder name
DEPLOYMENT = "data/2025-01-29/"
sessions = list_prefixes(prefix=DEPLOYMENT)
SESSION = next((s for s in sessions if "green" in s), sessions[0])
print("Using session:", SESSION)

meteo_df = get_csv(f"{SESSION}metadata/meteo_data.csv")
snow_df  = get_csv(f"{SESSION}metadata/snow_data.csv")

print("Meteo data columns:", list(meteo_df.columns))
print(f"\nMeteo data — {len(meteo_df)} rows")
display(meteo_df.head())

print(f"\nSnow data — {len(snow_df)} rows")
display(snow_df.head())

### 3b. Loading IMU data

IMU data from both the VectorNav VN-100 and Xsens MTi-30 are stored as CSV files with columns `[t, wx, wy, wz, ax, ay, az]`, where `t` is a UNIX timestamp in microseconds and the remaining fields are angular velocities (rad/s) and linear accelerations (m/s²).

In [ ]:
imu_df = get_csv(f"{SESSION}vectornav.csv")

print("IMU columns:", list(imu_df.columns))
print(f"Total IMU samples: {len(imu_df)}")
display(imu_df.head())

### 3c. Loading lidar point clouds (`.bin`)

Lidar scans are stored as raw binary float32 files. Each point has **6 fields**: `[x, y, z, intensity, ring_id, timestamp]`. The filename corresponds to the UNIX timestamp (in microseconds) of the **first point** in the scan.

> **Note:** Lidar data is **not** motion-compensated. The SDK provides scripts for ego-motion compensation if needed.

> 📖 **See also:** The [Lidar Data Example](https://github.com/norlab-ulaval/fomo-sdk) tutorial in the SDK covers loading, visualization, top-view rendering, and `.bin` ↔ `.csv` conversion, in both **Python and Rust**.

In [ ]:
def load_lidar_scan(key):
    """
    Load a FoMo lidar point cloud from S3.
    Returns an (N, 6) float32 array: [x, y, z, intensity, ring_id, timestamp]
    Each point is 6 float32 values = 24 bytes.
    """
    obj = s3.get_object(Bucket=BUCKET, Key=key)
    data = obj["Body"].read()
    # Truncate to nearest multiple of 24 bytes (6 float32 values per point)
    point_size = 4 * 6  # 4 bytes per float32, 6 fields per point
    remainder = len(data) % point_size
    if remainder != 0:
        data = data[:len(data) - remainder]
    raw = np.frombuffer(data, dtype=np.float32)
    return raw.reshape(-1, 6)

# List the first few scans in a session to get a valid filename
lidar_prefix = f"{SESSION}robosense/"
resp = s3.list_objects_v2(Bucket=BUCKET, Prefix=lidar_prefix, MaxKeys=3)
first_scan_key = resp["Contents"][0]["Key"]

print("Loading scan:", first_scan_key)
points = load_lidar_scan(first_scan_key)
print(f"Point cloud shape: {points.shape}  →  {points.shape[0]:,} points")
print("\nFirst 5 points (x, y, z, intensity, ring_id, timestamp):")
print(points[:5])

### 3d. Loading radar scans (`.png`)

Navtech radar scans follow the **Oxford RobotCar** polar image convention. Each image has `M` rows (azimuths) and `11 + R` columns, where:
- **Columns 0–7**: 64-bit azimuth timestamp (encoded as 8 uint8 bytes)
- **Columns 8–9**: 16-bit rotational encoder value
- **Column 10**: empty (reserved for Oxford format compatibility)
- **Columns 11 onwards**: `R` range bins (intensity values)

The SDK provides a utility to convert from polar to Cartesian coordinates.

> 📖 **See also:** The [Radar Data Example](https://github.com/norlab-ulaval/fomo-sdk) tutorial demonstrates polar visualization, polar ↔ Cartesian conversion, and combined lidar+radar overlays, in both **Python and Rust**.

In [ ]:
from PIL import Image

def load_radar_scan(key):
    """Load a Navtech radar polar image from S3 as a numpy array."""
    obj = s3.get_object(Bucket=BUCKET, Key=key)
    img = Image.open(io.BytesIO(obj["Body"].read()))
    return np.array(img)

radar_prefix = f"{SESSION}navtech/"
resp = s3.list_objects_v2(Bucket=BUCKET, Prefix=radar_prefix, MaxKeys=3)
first_radar_key = resp["Contents"][0]["Key"]

radar_img = load_radar_scan(first_radar_key)
print(f"Radar image shape: {radar_img.shape}  →  {radar_img.shape[0]} azimuths × {radar_img.shape[1]} columns")
print(f"Range bins (R): {radar_img.shape[1] - 11}")

### 3e. Loading the Ground Truth trajectory

Ground truth is provided in the **TUM format**: `t x y z qx qy qz qw`. Positions `(x, y, z)` are derived from post-processed kinematic (PPK) GNSS using three receivers mounted on the robot. Orientation is set to identity quaternion as heading estimation is not provided.

In [ ]:
def load_ground_truth(session_prefix):
    """Load the TUM-format ground truth trajectory."""
    obj = s3.get_object(Bucket=BUCKET, Key=f"{session_prefix}gt.txt")
    content = obj["Body"].read().decode("utf-8")
    rows = []
    for line in content.strip().splitlines():
        if line.startswith("#") or not line.strip():
            continue
        rows.append([float(v) for v in line.split()])
    return pd.DataFrame(rows, columns=["t", "x", "y", "z", "qx", "qy", "qz", "qw"])

gt_df = load_ground_truth(SESSION)
print(f"Ground truth: {len(gt_df)} poses")
display(gt_df.head())

# Trajectory length in metres (approximate)
dx = np.diff(gt_df["x"].values)
dy = np.diff(gt_df["y"].values)
total_dist = np.sum(np.sqrt(dx**2 + dy**2))
print(f"\nApproximate trajectory length: {total_dist:.1f} m")

### 3f. Loading calibration files

Each session contains a `calib/` folder with JSON files for all sensors. These include camera intrinsics (focal lengths, principal point, distortion) and extrinsics (rigid-body transforms between sensor frames).

In [ ]:
def load_calib(session_prefix, filename):
    obj = s3.get_object(Bucket=BUCKET, Key=f"{session_prefix}calib/{filename}")
    return json.loads(obj["Body"].read().decode("utf-8"))

transforms = load_calib(SESSION, "transforms.json")
print("Available transforms (sensor frame pairs):")
if isinstance(transforms, dict):
    pprint(list(transforms.keys()))
elif isinstance(transforms, list):
    pprint(transforms)
else:
    pprint(transforms)

### 3g. Loading camera images and audio

Camera images (Basler, ZED X left/right) are stored as standard `.png` files and can be loaded with any image library (e.g., `PIL`, `cv2`). Audio data is stored as 1-second mono `.wav` snippets at 16 kHz for each microphone channel.

> 📖 **See also:** The SDK provides dedicated Python tutorials for these modalities:
> - [Image Data Example](https://github.com/norlab-ulaval/fomo-sdk) — loading, visualization, and lidar point projection onto camera images.
> - [Audio Data Example](https://github.com/norlab-ulaval/fomo-sdk) — loading stereo audio, spectrogram generation, and WAV export.

---
## Q4: Visuals — a picture is worth a thousand words

Let's create several visualizations to illustrate what makes the FoMo dataset unique.

### 4a. Ground truth trajectory — top-down view

We first visualize the 2D path of the robot for a single session using the PPK-GNSS ground truth.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 7), dpi=100, facecolor="white")

# Normalize to local coordinates
x = gt_df["x"] - gt_df["x"].iloc[0]
y = gt_df["y"] - gt_df["y"].iloc[0]

# Color by time progression
t_norm = (gt_df["t"] - gt_df["t"].iloc[0]) / (gt_df["t"].iloc[-1] - gt_df["t"].iloc[0])
sc = ax.scatter(x, y, c=t_norm, cmap="viridis", s=2, zorder=2)
ax.plot(x.iloc[0], y.iloc[0], "go", markersize=10, label="Start", zorder=3)
ax.plot(x.iloc[-1], y.iloc[-1], "rs", markersize=10, label="End", zorder=3)

cbar = plt.colorbar(sc, ax=ax, shrink=0.6)
cbar.set_label("Time progression", fontsize=10)

ax.set_xlabel("X (m)", fontsize=11)
ax.set_ylabel("Y (m)", fontsize=11)
ax.set_title("Ground Truth Trajectory\nGreen – 2025-01-29 (Winter, −19 °C)",
             fontsize=13, fontweight="bold", pad=12)
ax.set_aspect("equal")
ax.legend(fontsize=10)
ax.grid(True, linestyle="--", alpha=0.3)
ax.set_facecolor("#f8f9fa")
for spine in ["top", "right"]:
    ax.spines[spine].set_visible(False)

plt.tight_layout()
plt.show()

### 4b. Lidar point cloud — top-down bird's-eye view

A single RoboSense Ruby Plus scan projected into the XY plane, colored by height (Z).

In [ ]:
# Filter ground points and distant returns for a cleaner view
mask = (
    (np.abs(points[:, 0]) < 40) &
    (np.abs(points[:, 1]) < 40) &
    (points[:, 2] > -2.0) &
    (points[:, 2] < 5.0)
)
pts = points[mask]

fig, ax = plt.subplots(figsize=(8, 8), dpi=100, facecolor="#0d0d0d")
sc = ax.scatter(pts[:, 0], pts[:, 1], c=pts[:, 2],
                cmap="plasma", s=0.3, alpha=0.8, vmin=-1.5, vmax=3.0)

cbar = plt.colorbar(sc, ax=ax, shrink=0.5)
cbar.set_label("Height Z (m)", color="white", fontsize=10)
cbar.ax.yaxis.set_tick_params(color="white")
plt.setp(cbar.ax.get_yticklabels(), color="white")

ax.set_xlabel("X (m)", color="white", fontsize=11)
ax.set_ylabel("Y (m)", color="white", fontsize=11)
ax.set_title("RoboSense Ruby Plus — Single Scan (BEV)\nGreen – 2025-01-29",
             color="white", fontsize=13, fontweight="bold", pad=12)
ax.set_aspect("equal")
ax.set_facecolor("#0d0d0d")
ax.tick_params(colors="white")
for spine in ax.spines.values():
    spine.set_edgecolor("#444")

plt.tight_layout()
plt.show()
print(f"Plotted {pts.shape[0]:,} points (out of {points.shape[0]:,} total)")

### 4c. Radar scan — polar image

A raw Navtech CIR-304H scan displayed as a polar image. Brighter pixels indicate stronger radar returns.

In [ ]:
# Extract only the range bins (columns 11 onwards)
range_data = radar_img[:, 11:].astype(np.float32)

fig, ax = plt.subplots(figsize=(10, 4), dpi=100, facecolor="white")
im = ax.imshow(range_data, aspect="auto", cmap="inferno",
               interpolation="nearest", vmin=0, vmax=255)
plt.colorbar(im, ax=ax, shrink=0.8, label="Intensity")

ax.set_xlabel("Range bin", fontsize=11)
ax.set_ylabel("Azimuth index", fontsize=11)
ax.set_title("Navtech CIR-304H — Radar Polar Image\nGreen – 2025-01-29",
             fontsize=13, fontweight="bold", pad=12)

plt.tight_layout()
plt.show()
print(f"Polar image: {range_data.shape[0]} azimuths × {range_data.shape[1]} range bins")

### 4d. Seasonal temperature & snow depth overview

One of FoMo's unique strengths is its complete seasonal coverage. Below we plot the meteorological conditions across the 12 deployments to give a sense of the environmental variety.

In [ ]:
# Approximate daily averages per deployment (from the paper / dataset documentation)
deployment_info = [
    {"date": "Nov 21",  "temp": -3,  "snow_cm": 24,  "conditions": "Clouds"},
    {"date": "Nov 28",  "temp": -5,  "snow_cm": 40,  "conditions": "Clear"},
    {"date": "Jan 10",  "temp": -15, "snow_cm": 60,  "conditions": "Clear"},
    {"date": "Jan 29",  "temp": -19, "snow_cm": 72,  "conditions": "Clear"},
    {"date": "Mar 10",  "temp": -8,  "snow_cm": 120, "conditions": "Snowfall"},
    {"date": "Apr 15",  "temp": 4,   "snow_cm": 30,  "conditions": "Rain"},
    {"date": "May 28",  "temp": 12,  "snow_cm": 0,   "conditions": "Clear"},
    {"date": "Jun 26",  "temp": 18,  "snow_cm": 0,   "conditions": "Clear"},
    {"date": "Aug 20",  "temp": 18,  "snow_cm": 0,   "conditions": "Clear"},
    {"date": "Sep 24",  "temp": 10,  "snow_cm": 0,   "conditions": "Clear"},
    {"date": "Oct 14",  "temp": 5,   "snow_cm": 0,   "conditions": "Clear"},
    {"date": "Nov 03",  "temp": 2,   "snow_cm": 0,   "conditions": "Night/Rain"},
]
df_meta = pd.DataFrame(deployment_info)
dates = df_meta["date"]
x = np.arange(len(dates))

fig, ax1 = plt.subplots(figsize=(13, 5), dpi=100, facecolor="white")
ax2 = ax1.twinx()

# Temperature bars (color by value)
colors = ["#3b82f6" if t < 0 else "#f97316" for t in df_meta["temp"]]
ax1.bar(x - 0.2, df_meta["temp"], width=0.35, color=colors, alpha=0.85,
        label="Avg. Temperature (°C)", zorder=2)

# Snow depth line
ax2.plot(x, df_meta["snow_cm"], "o--", color="#60a5fa", linewidth=2,
         markersize=7, label="Snow Depth (cm)", zorder=3)
ax2.fill_between(x, df_meta["snow_cm"], alpha=0.12, color="#60a5fa")

ax1.axhline(0, color="black", linewidth=0.7, linestyle="-")
ax1.set_xticks(x)
ax1.set_xticklabels(dates, rotation=30, ha="right", fontsize=10)
ax1.set_ylabel("Temperature (°C)", fontsize=11)
ax2.set_ylabel("Snow Depth (cm)", fontsize=11, color="#3b82f6")
ax2.tick_params(axis="y", colors="#3b82f6")
ax1.set_title("FoMo Deployment Conditions — Temperature & Snow Depth",
              fontsize=13, fontweight="bold", pad=12)

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc="upper left", fontsize=10)

ax1.set_facecolor("#f8f9fa")
ax1.grid(True, linestyle="--", alpha=0.3, axis="y")
for spine in ["top"]:
    ax1.spines[spine].set_visible(False)

plt.tight_layout()
plt.show()

### 4e. Power consumption across seasons

The dataset includes detailed power monitoring. Current draw from the motors is an indirect indicator of terrain difficulty — higher current means harder terrain (e.g., deep snow, steep slopes). Let's explore the current readings for the January deployment.

In [ ]:
current_left_df  = get_csv(f"{SESSION}metadata/current_left.csv")
current_right_df = get_csv(f"{SESSION}metadata/current_right.csv")

# Assume columns are [timestamp, current_A]
t_col = current_left_df.columns[0]
c_col = current_left_df.columns[1]

t_left  = current_left_df[t_col].values
t_right = current_right_df[t_col].values
# Normalize time to seconds from start
t0 = min(t_left[0], t_right[0])
t_left_s  = (t_left  - t0) / 1e6
t_right_s = (t_right - t0) / 1e6

fig, ax = plt.subplots(figsize=(12, 4), dpi=100, facecolor="white")
ax.plot(t_left_s,  current_left_df[c_col],  label="Left motor",  alpha=0.8, linewidth=0.8)
ax.plot(t_right_s, current_right_df[c_col], label="Right motor", alpha=0.8, linewidth=0.8)

ax.set_xlabel("Time (s)", fontsize=11)
ax.set_ylabel("Current (A)", fontsize=11)
ax.set_title("Motor Current Draw — Green Trajectory, Jan 29 2025 (−19 °C, Tracks)",
             fontsize=12, fontweight="bold", pad=10)
ax.legend(fontsize=10)
ax.grid(True, linestyle="--", alpha=0.3)
ax.set_facecolor("#f8f9fa")
for spine in ["top", "right"]:
    ax.spines[spine].set_visible(False)

plt.tight_layout()
plt.show()

print(f"Peak current — Left:  {current_left_df[c_col].max():.1f} A")
print(f"Peak current — Right: {current_right_df[c_col].max():.1f} A")

---
## Q5: One question we have answered using these data

### How robust are state-of-the-art localization methods to seasonal changes in a boreal forest?

This is the central research question explored in the accompanying paper (Boxan et al., 2026). Using the FoMo dataset, we evaluated four methods — **Lidar-Inertial Odometry**, **Radar-Gyro Teach & Repeat**, **Stereo-Inertial SLAM (ORB-SLAM3)**, and a simple **Proprioceptive baseline** — across all 12 deployments.

For each pair of deployments, one was used to build a map and the other to localize within it. The results are summarized in a **translation drift matrix** (mean drift in %).

The key finding: **seasonal changes significantly degrade localization accuracy**, particularly for lidar-based methods when snowbanks drastically alter the scene geometry. The January 29, 2025 and March 10, 2025 deployments (>1 m of snow) caused mean translation drifts exceeding 40% for the Lidar-Inertial method — and even a complete localization failure in one case. ORB-SLAM3 (visual SLAM), which benefits from loop closure, showed the most consistent performance, with errors below 2% in many cases.

Below we reproduce the summary comparison table from Table 3 of the paper.

In [ ]:
# Translation drift [%] for each method when mapping deployment = Jan 29, 2025
# Source: Table 3, Boxan et al. (2026)
deployments_eval = ["Nov21", "Nov28", "Jan29", "Mar10", "Jun26", "Aug20", "Sep24", "Oct14", "Nov03"]

results = {
    "Proprioceptive":       [3.7,  4.6,  15.3, 4.6,  3.9, 3.4,  5.0,  3.7,  4.6],
    "Lidar-Inertial":       [4.5,  4.8,  3.8,  4.3,  11.7, 4.0, 25.4, 46.5, 4.3],
    "Radar-Gyro T&R":       [8.1, 14.2,  16.8, 32.3,  8.8, 4.2,  9.3, 24.6, 9.3],
    "Stereo-Inertial SLAM": [1.2,  1.5,  2.4,  7.6,   8.8, 1.3,  2.3,  0.9, None],  # Nov03: N/A (night)
}

df_results = pd.DataFrame(results, index=deployments_eval)

fig, ax = plt.subplots(figsize=(13, 5), dpi=100, facecolor="white")

x = np.arange(len(deployments_eval))
width = 0.2
colors_methods = ["#6366f1", "#f59e0b", "#10b981", "#ef4444"]

for i, (method, color) in enumerate(zip(df_results.columns, colors_methods)):
    vals = [v if v is not None else 0 for v in df_results[method]]
    bars = ax.bar(x + i * width - 1.5 * width, vals, width,
                  label=method, color=color, alpha=0.85, zorder=2)
    # Mark N/A
    for j, v in enumerate(df_results[method]):
        if v is None:
            ax.text(x[j] + i * width - 1.5 * width, 2, "N/A",
                    ha="center", va="bottom", fontsize=7, color="gray", rotation=90)

ax.axhline(5, color="gray", linestyle="--", linewidth=0.8, alpha=0.6, label="5% drift threshold")
ax.set_xticks(x)
ax.set_xticklabels(deployments_eval, fontsize=10)
ax.set_ylabel("Mean Translation Drift (%)", fontsize=11)
ax.set_title("Localization Performance Across Deployments\n(Map built from Jan 29, 2025)",
             fontsize=13, fontweight="bold", pad=12)
ax.legend(fontsize=9, loc="upper left")
ax.set_facecolor("#f8f9fa")
ax.grid(True, linestyle="--", alpha=0.3, axis="y")
for spine in ["top", "right"]:
    ax.spines[spine].set_visible(False)

plt.tight_layout()
plt.show()

---
## Q6: One unanswered question — the community challenge

### Can a machine learning model predict terrain traversability and energy cost from multi-modal sensor data in a boreal forest across all seasons?

The FoMo dataset contains the ingredients necessary to tackle this open problem:

- **Input features:** lidar point clouds, stereo images, radar scans, and IMU data providing rich, multi-modal environmental descriptions.
- **Labels / supervision signal:** motor current draw (`current_left.csv`, `current_right.csv`) and battery logs (`battery_logs.csv`) provide direct measures of energy expenditure. Immobilization events (robot stuck in snow) provide hard traversability failures.
- **Seasonal variation:** the same physical locations are observed under radically different conditions (bare ground, fresh snow, packed snowbanks, spring mud, summer vegetation). A model that generalizes across seasons must learn geometry- and appearance-invariant terrain representations.

**Recommended starting approach:**

1. **Define the task:** regress motor current from sensor observations to predict instantaneous energy cost, or classify terrain into traversability categories (easy / hard / immobilized).
2. **Align data modalities:** use the provided calibration files and timestamps to synchronize lidar, camera, and metadata at a common frequency (e.g., 10 Hz, matching the lidar rate).
3. **Leverage the off-trail trajectories:** the Green and Magenta trajectories offer the greatest terrain challenge diversity and are ideal test beds for traversability models.
4. **Cross-season evaluation:** train on summer/autumn deployments and evaluate on winter deployments (and vice versa) to measure the model's robustness to appearance change.
5. **Baseline comparison:** compare against simple elevation-based traversability (from lidar) to measure the added value of visual and radar inputs.



---
## Downloading a full session

To download an entire trajectory session locally using the AWS CLI (no account required):

```bash
# Download the complete Red trajectory from Jan 29, 2025
aws s3 cp s3://fomo-dataset/data/2025-01-29/red_2025-01-30-09-58/ ./fomo_red_jan29/ \\
    --recursive --no-sign-request
```

For users who only need a subset of modalities (e.g., lidar + IMU only), the dataset download backend on the website supports **selective modality downloads**, avoiding unnecessary data transfer for the full >10 TB dataset.

The **SDK** (Python + Rust) for data conversion, calibration utilities, and ROS2 mcap generation is available at [https://github.com/norlab-ulaval/fomo-sdk](https://github.com/norlab-ulaval/fomo-sdk).

The SDK includes the following tutorials to help you get started:

| Tutorial | Python | Rust |
|----------|:------:|:----:|
| Audio data | ✅ | — |
| Image data | ✅ | — |
| Lidar data (`.bin` ↔ `.csv`) | ✅ | ✅ |
| Radar data (polar ↔ Cartesian) | ✅ | ✅ |
| Evaluation (SLAM) | ✅ | — |

> **Note:** Rust support is explicitly documented for lidar and radar data conversion. Audio and image tutorials are currently Python-only. Check [https://github.com/norlab-ulaval/fomo-sdk](https://github.com/norlab-ulaval/fomo-sdk) for the latest updates.

---
## Summary

| Attribute | Value |
|-----------|-------|
| **Total distance** | >64 km |
| **Deployments** | 12 (Nov 2024 – Nov 2025) |
| **Trajectories** | 6 (Red, Blue, Green, Magenta, Orange, Yellow) |
| **Temperature range** | −19 °C to +18 °C (37 °C spread) |
| **Max snow depth** | >1 m in open areas; >3 m roadside snowbanks |
| **Sensors** | 2× lidar, 1× FMCW radar, stereo camera, mono camera, 2× IMU, 2× microphone, GNSS |
| **Ground truth** | PPK-GNSS, 3 receivers, per-point covariances, TUM format |
| **Bucket** | `s3://fomo-dataset` (public, no credentials needed) |
| **License** | See `s3://fomo-dataset/LICENSE.txt` |
| **Website & SDK** | [fomo.norlab.ulaval.ca](https://fomo.norlab.ulaval.ca) |

Questions, bug reports, and contributions are welcome via the [dataset GitHub repository](https://github.com/norlab-ulaval/fomo-dataset).